# A1 cross-subject re-ID — EEGNet (with volts→microvolts fix)

Runs experiment `02_closed_set_reid` for all three victim families on the full 104-subject PhysioNet motor-imagery split. Designed to take ~25–30 minutes on a Colab L4. Total session disk use: ~3 GB (1.7 GB EDFs + 2.3 GB cached windows).

After Cell 7 finishes, copy the entire stdout from Cell 6 and paste it back to the chat. I'll update the repo's `results/02_closed_set_reid.json` and the figure from there.

**Runtime → L4 GPU.** Set via Runtime → Change runtime type → L4 GPU.

## 1. Clone the repo (always pulls latest `main`)

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install project deps

Colab already ships with torch / numpy / scipy / pandas / scikit-learn / matplotlib. We only install the EEG-specific packages and force the matching torchaudio (Colab's preinstalled torchaudio is sometimes a different ABI build).

In [ ]:
import torch
tv = torch.__version__.split('+')[0]   # e.g. '2.5.1'
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2–3 min on archive mirror)

The archive mirror (`archive.physionet.org`) serves the EDFs ~10× faster than the versioned URL on `physionet.org`. Total download: 1.35 GB × 11 MB/s ≈ 2 minutes.

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run A1 cross-subject re-ID — all three victim families

EEGNet now uses `input_scale=1e6` to convert volts → microvolts before the network's first conv. The previous chance-level EEGNet result was caused by gradients vanishing at the volt scale; this run replaces that EEGNet row of the A1 table.

Expected wall: ~25 min (EEGNet train ~8 min on L4, FBCSP ~7 min, Riemann ~1 min, attacks ~5 min).

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.02_closed_set_reid --all

## 6. Show the result JSON and figure path

Copy the JSON contents and the markdown table back to the chat. I'll commit them to the repo from there.

In [ ]:
print("=== JSON ===")
with open('results/02_closed_set_reid.json') as f:
    print(f.read())
print()
print("=== figure ===")
!ls -la figures/02_closed_set_reid.pdf

## 7. Download the figure (optional — only if you want a copy locally)

In [ ]:
from google.colab import files
files.download('results/02_closed_set_reid.json')
files.download('figures/02_closed_set_reid.pdf')